# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Task type: Ranking / Scoring

I'm calling this a **ranking** problem with a **scoring** model underneath.

The real business question is: "Out of thousands of pages, which one should an editor look at first?" That's a ranking question. Score each page, sort by score, hand the editor a prioritized list. They start at the top and work down until they run out of time.

This isn't classification — we're not sorting pages into "fix" vs "don't fix" buckets (many pages need varying levels of attention). It's not clustering either — I don't need to discover groups of similar pages.

**Why ranking:**
- Editors have limited time — an unordered list is useless
- Not all mistakes cost the same: wasting time on page #50 is annoying, wasting it on page #1 is a real problem
- The top of the queue has to be genuinely urgent — high precision at the top is the whole point

In [13]:
import pandas as pd
import numpy as np

# Confirm the data shape
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Dataset: {len(df):,} pages, {df['client_id'].nunique()} clients")
print(f"Trend distribution:\n{df['trend_direction'].value_counts().to_string()}")

Dataset: 30,000 pages, 32 clients
Trend distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target: Decline flag (proxy — it's a rule, but it's what we have)

Honestly: the label is **`trend_direction == "down"`**. That's not a truly observed outcome — it's computed from `trend_pct` (if trend_pct drops below a threshold, direction flips to "down"). So it IS a rule. The data dictionary warns about this explicitly.

The ideal would be: take a snapshot now, wait 30 days, check what actually declined. But this dataset is a single snapshot — there's no future window. So I'm using the computed flag as a **proxy**.

**Why I'm still using it:**
- It's the closest thing we have to "which pages actually decline" in this dataset
- It maps directly to the decision: pages flagged as declining should get reviewed first
- I just need to be honest about what it is (a proxy) and not treat it like a ground-truth observed outcome

**What I'll watch out for:** The model should NEVER see `trend_direction` or `trend_pct` as features — that would be textbook leakage (the model would literally see the answer). The `is_declining_label` column exists only as the target, separated out in the prep step.

In [14]:
# Show what the target looks like in the data
target_counts = df['trend_direction'].value_counts()
target_pcts = df['trend_direction'].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    'count': target_counts,
    'percent': target_pcts
})
print("Target distribution (current trend_direction as decline proxy):")
print(target_summary.round(1).to_string())

# Binary flag: "needs attention" = declining or flat
df['needs_attention'] = df['trend_direction'].isin(['down', 'flat'])
print(f"\nBinary proxy (down + flat = needs attention): {df['needs_attention'].sum():,} pages ({df['needs_attention'].mean()*100:.1f}%)")

Target distribution (current trend_direction as decline proxy):
                 count  percent
trend_direction                
down             16262     54.2
stable            5962     19.9
up                4388     14.6
new               2236      7.5
flat              1152      3.8

Binary proxy (down + flat = needs attention): 17,414 pages (58.0%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Primary metric: Precision@K (P@100)

I'm going with **P@100** — of the top 100 pages the model recommends, how many actually need attention?

**Why P@100:**
- An editor works through one queue per cycle, maybe 50–200 pages. The top has to be right.
- It answers the direct business question: "Is the model wasting the editor's first batch?"
- Wrong calls at rank 1 hurt way more than at rank 500 — P@100 captures that asymmetry

**Baseline:** The existing hand-written rules have a certain P@100. The model needs to beat that — if the rules already get 70% precision at the top, I need to exceed it.

**Secondary metric (for diagnostics):** Mean reciprocal rank (MRR) — how early in the queue does the first truly urgent page appear?

**What "good" looks like:** P@100 beats the baseline by a clear margin (say +5 points), confirmed with time-aware cross-validation.

In [15]:
# Estimate baseline P@100 from available signals
# If we used a simple rule: "flag pages with avg_position > 10"
simple_rule = df[df['avg_position'] > 10]
top_of_queue = 100

# How many of the "worst position pages" actually need attention?
df_sorted = df.sort_values('avg_position', ascending=False)
top_k = df_sorted.head(top_of_queue)
pat_k = top_k['needs_attention'].mean()
print(f"Baseline P@{top_of_queue} using 'worst position first' rule:")
print(f"  {pat_k:.1%} of the top {top_of_queue} pages need attention")

# What's the overall base rate?
base_rate = df['needs_attention'].mean()
print(f"  Overall base rate: {base_rate:.1%}")
print(f"  A random top-{top_of_queue} would get ~{base_rate:.0%} precision")

Baseline P@100 using 'worst position first' rule:
  42.0% of the top 100 pages need attention
  Overall base rate: 58.0%
  A random top-100 would get ~58% precision


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [16]:
# Unit of analysis: one row = one content page
df['needs_attention'] = df['trend_direction'].isin(['down', 'flat'])

print(f"Unit of analysis: one row = one content page")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

# Show key columns for the scoring task
key_cols = [
    'content_id', 'client_id',
    'impressions_90d', 'clicks_90d', 'ctr', 'avg_position',
    'pageviews_90d', 'sessions_90d', 'engagement_rate',
    'content_age_days', 'days_since_last_update',
    'trend_direction', 'trend_pct',
    'impression_tier', 'position_tier'
]
print("Key columns for scoring:\n")
df[key_cols].head(10)

Unit of analysis: one row = one content page
Shape: 30,000 rows × 45 columns

Key columns for scoring:



,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,pageviews_90d,sessions_90d,engagement_rate,content_age_days,days_since_last_update,trend_direction,trend_pct,impression_tier,position_tier
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,22,17,5.88,187,20,down,-41.4,good,striking
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,10,9,0.00,445,25,down,-57.7,good,page_3_5
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,14,11,0.00,141,20,down,-60.9,good,page_3_5
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,87,78,1.28,463,22,stable,-13.8,good,page_1
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,177,145,0.00,263,14,down,-34.7,good,page_3_5
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.03,8.5,4,5,0.00,147,20,down,-38.9,good,page_1
6,content_9a34b442b552,client_8722616204,20,0,0.00,7.0,1,1,0.00,90,20,down,-92.3,low,page_1
7,content_a63219c6e95a,client_19581e27de,1724,1,0.06,21.2,28,28,3.57,445,22,stable,0.6,moderate,page_3_5
8,content_5e6c160719bc,client_6208ef0f77,32574,29,0.09,46.0,128,68,5.88,90,20,down,-58.8,excellent,page_3_5
9,content_c27558df2b0c,client_19581e27de,1240,2,0.16,4.9,4,3,0.00,257,104,down,-29.2,moderate,page_1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### A fixed rule isn't enough

A hand-written rule like "flag pages where avg_position > 10 AND ctr < 0.5%" misses a lot:

1. **Signals interact in weird ways.** A page at position 5 with dropping CTR is more urgent than one at position 20 with stable CTR. A linear rule can't capture that tradeoff.

2. **Many weak signals add up.** Content age, days since last update, engagement rate, scroll depth, AI traffic — each is weak alone but informative together. Writing if-statements for 15+ signals is brittle and impossible to maintain.

3. **Patterns shift.** What "needs attention" looks different after a Google update, during holidays, or as content ages. A fixed rule drifts. A model gets retrained.

4. **Edge cases kill thresholds.** Pages with high impressions but low CTR (lost snippet?) look totally different from pages with low impressions and low CTR (irrelevant topic?). One threshold can't cover both.

**Bottom line:** A rule is simple to understand but it plateaus. ML can combine more signals, capture those interactions, and adapt when things change. That's exactly what we need here.

In [17]:
# Demonstrate: single-rule vs multi-signal separation

# Rule 1: avg_position > 10
rule1_precision = df[df['avg_position'] > 10]['needs_attention'].mean()
rule1_coverage = (df['avg_position'] > 10).mean()

# Rule 2: ctr < 0.5%
rule2_precision = df[df['ctr'] < 0.5]['needs_attention'].mean()
rule2_coverage = (df['ctr'] < 0.5).mean()

# Combine both rules (AND)
combined = df[(df['avg_position'] > 10) & (df['ctr'] < 0.5)]
combined_precision = combined['needs_attention'].mean()
combined_coverage = len(combined) / len(df)

print("Fixed rules vs. their separation power:")
print(f"  Rule 1 (avg_position > 10):               coverage={rule1_coverage:.1%}, precision={rule1_precision:.1%}")
print(f"  Rule 2 (ctr < 0.5%):                     coverage={rule2_coverage:.1%}, precision={rule2_precision:.1%}")
print(f"  Combined (AND):                           coverage={combined_coverage:.1%}, precision={combined_precision:.1%}")
print(f"  Base rate:                                {df['needs_attention'].mean():.1%}")
print()
print("ML can capture interactions these thresholds miss — e.g., a page at position 5")
print("with dropping CTR behaves like a position-20 page, but thresholds treat them separately.")

Fixed rules vs. their separation power:
  Rule 1 (avg_position > 10):               coverage=52.7%, precision=58.1%
  Rule 2 (ctr < 0.5%):                     coverage=85.8%, precision=59.4%
  Combined (AND):                           coverage=47.4%, precision=58.8%
  Base rate:                                58.0%

ML can capture interactions these thresholds miss — e.g., a page at position 5
with dropping CTR behaves like a position-20 page, but thresholds treat them separately.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.